# 07 · 绩效归因

> 本 notebook 是《量化研究入门学习资料》第 X 章的可运行配套。
> 数据源：`data/csv/`。运行前请先执行 `python scripts/generate_data.py` 生成数据。

## 目标
重算 Brinson 归因（配置/选股效应）与风格暴露，对照 `attribution.json`。

In [ ]:
import pandas as pd, numpy as np, json
factors = pd.read_csv("data/csv/factors.csv", parse_dates=["date"])
basic = pd.read_csv("data/csv/stocks_basic.csv")
FACTORS = ["EP", "SIZE", "MOM60", "REV5", "VOL20", "TURN", "ROE", "GROW"]
w = np.full(8, 1 / 8)
INDUSTRIES = ["银行", "非银金融", "医药生物", "电子", "食品饮料",
              "机械设备", "汽车", "电力公用"]
ind_map = dict(zip(basic["code"], basic["industry"]))

# ---- 行业收益 / 组合行业权重 / 基准权重 ----
ind_ret, w_p, w_b, r_p = {k: [] for k in INDUSTRIES}, [], [], []
for _, g in factors.groupby("date"):
    y = g["next_return"].values
    alive = y == y
    if not alive.any():                                  # 最后一期无标签，平走
        for k in INDUSTRIES: ind_ret[k].append(0.0)
        w_p.append(np.zeros(len(INDUSTRIES))); w_b.append(np.zeros(len(INDUSTRIES)))
        r_p.append(0.0); continue
    inds_g = g["code"].map(ind_map).values              # 单期截面行业
    for k in INDUSTRIES:
        mm = (inds_g == k) & alive
        ind_ret[k].append(float(y[mm].mean()) if mm.sum() else 0.0)
    z = g[[f"z_{f}" for f in FACTORS]].fillna(0.0).values @ w
    order = np.argsort(-z); order = order[y[order] == y[order]]
    top = order[:20]
    wp = np.zeros(len(INDUSTRIES)); wb = np.zeros(len(INDUSTRIES))
    for i in top: wp[INDUSTRIES.index(ind_map[g["code"].iloc[i]])] += 1 / 20
    for i in np.where(alive)[0]: wb[INDUSTRIES.index(ind_map[g["code"].iloc[i]])] += 1 / alive.sum()
    w_p.append(wp); w_b.append(wb); r_p.append(float(y[top].mean()))

ir_ = np.array([ind_ret[k] for k in INDUSTRIES]).T
w_p, w_b = np.array(w_p), np.array(w_b)
alloc = (w_p - w_b) * ir_
total = np.array(r_p) - (w_b * ir_).sum(1)
sel = total - alloc.sum(1)

# 季度聚合
ddates = [str(d.date()) for d in factors["date"].unique()]
q = pd.PeriodIndex(ddates, freq="Q")
agg = pd.DataFrame({"alloc": alloc.sum(1), "sel": sel, "total": total}, index=q).groupby(level=0).sum()
print("前 4 季度归因：")
print(agg.head(4).round(4))

In [ ]:
# ---- 对照 attribution.json ----
ref = json.load(open("data/attribution.json", encoding="utf-8"))
ok_a = np.allclose(agg["alloc"].values, ref["brinson"]["allocation"], atol=1e-3)
ok_s = np.allclose(agg["sel"].values, ref["brinson"]["selection"], atol=1e-3)
print("配置效应对照:", "PASS" if ok_a else "FAIL")
print("选股效应对照:", "PASS" if ok_s else "FAIL")
assert ok_a and ok_s

# ---- 风格暴露（组合持仓的因子加权 z）----
style = {f: [] for f in ["SIZE", "MOM60", "VOL20", "EP", "TURN"]}
for _, g in factors.groupby("date"):
    y = g["next_return"].values
    alive = y == y
    if not alive.any():                            # 最后一期无标签，暴露记 0
        for f in style: style[f].append(0.0)
        continue
    z = g[[f"z_{f}" for f in FACTORS]].fillna(0.0).values @ w
    order = np.argsort(-z)
    order = order[alive[order]]                    # 过滤退市/无标签股票
    top = order[:20]
    for f in style: style[f].append(float(g[f"z_{f}"].values[top].mean()))
ok_st = all(np.allclose(np.array(style[f]), ref["style_exposure"][f], atol=1e-3) for f in style)
print("风格暴露对照:", "PASS" if ok_st else "FAIL")
assert ok_st